In [ ]:
!pip install requests pandas plotly -q

In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta

# Yesterday's date (Nord Pool always has yesterday's data)
yesterday = (datetime.today() - timedelta(days=1)).strftime("%Y-%m-%d")

# Fetch day-ahead prices for Sweden SE3
url = f"https://www.elprisetjustnu.se/api/v1/prices/{yesterday[:4]}/{yesterday[5:7]}-{yesterday[8:10]}_SE3.json"

response = requests.get(url)
data = response.json()

# Convert to DataFrame
df = pd.DataFrame(data)
df = df[["time_start", "SEK_per_kWh"]]
df["time_start"] = pd.to_datetime(df["time_start"])
df = df.rename(columns={"time_start": "hour", "SEK_per_kWh": "price_SEK_kWh"})

print(df)

                        hour  price_SEK_kWh
0  2026-05-19 00:00:00+02:00        1.10184
1  2026-05-19 00:15:00+02:00        1.08867
2  2026-05-19 00:30:00+02:00        1.07595
3  2026-05-19 00:45:00+02:00        1.05708
4  2026-05-19 01:00:00+02:00        1.06311
..                       ...            ...
91 2026-05-19 22:45:00+02:00        0.90951
92 2026-05-19 23:00:00+02:00        0.92125
93 2026-05-19 23:15:00+02:00        0.89679
94 2026-05-19 23:30:00+02:00        0.87978
95 2026-05-19 23:45:00+02:00        0.83908

[96 rows x 2 columns]


In [ ]:
import plotly.express as px

fig = px.line(
    df,
    x="hour",
    y="price_SEK_kWh",
    title="Nord Pool Day-Ahead Prices — Sweden SE3 (Yesterday)",
    labels={"hour": "Hour", "price_SEK_kWh": "Price (SEK/kWh)"}
)

fig.update_layout(
    template="plotly_dark",
    hovermode="x unified"
)

fig.show()

In [ ]:
# Resample to hourly prices
df_hourly = df.resample("h", on="hour").mean().reset_index()
df_hourly.columns = ["hour", "price"]
print(df_hourly)
print(f"\nHours: {len(df_hourly)}")

                        hour     price
0  2026-05-19 00:00:00+02:00  1.080885
1  2026-05-19 01:00:00+02:00  1.040675
2  2026-05-19 02:00:00+02:00  1.006720
3  2026-05-19 03:00:00+02:00  0.975835
4  2026-05-19 04:00:00+02:00  0.997313
5  2026-05-19 05:00:00+02:00  1.023725
6  2026-05-19 06:00:00+02:00  1.166047
7  2026-05-19 07:00:00+02:00  1.234755
8  2026-05-19 08:00:00+02:00  1.238158
9  2026-05-19 09:00:00+02:00  1.098083
10 2026-05-19 10:00:00+02:00  0.996023
11 2026-05-19 11:00:00+02:00  0.891770
12 2026-05-19 12:00:00+02:00  0.797662
13 2026-05-19 13:00:00+02:00  0.795195
14 2026-05-19 14:00:00+02:00  0.810858
15 2026-05-19 15:00:00+02:00  0.855315
16 2026-05-19 16:00:00+02:00  0.912260
17 2026-05-19 17:00:00+02:00  1.006252
18 2026-05-19 18:00:00+02:00  1.034203
19 2026-05-19 19:00:00+02:00  1.132065
20 2026-05-19 20:00:00+02:00  1.209933
21 2026-05-19 21:00:00+02:00  1.075098
22 2026-05-19 22:00:00+02:00  0.978002
23 2026-05-19 23:00:00+02:00  0.884225

Hours: 24


In [ ]:
!pip install pyomo highspy -q

In [ ]:
from pyomo.environ import *

# BESS Parameters
CAPACITY = 1.0      # MWh - battery size
MAX_POWER = 0.5     # MW  - max charge/discharge rate
EFFICIENCY = 0.90   # round-trip efficiency (90%)
INITIAL_SOC = 0.5   # start at 50% charge

prices = df_hourly["price"].values  # 24 hourly prices
T = len(prices)                      # 24 hours

# Build the model
model = ConcreteModel()

# Time steps
model.T = RangeSet(0, T-1)

# Decision variables
model.charge    = Var(model.T, bounds=(0, MAX_POWER))  # MW charged each hour
model.discharge = Var(model.T, bounds=(0, MAX_POWER))  # MW discharged each hour
model.soc       = Var(model.T, bounds=(0, CAPACITY))   # State of charge MWh

# Objective: maximize revenue from discharging, minus cost of charging
model.obj = Objective(
    expr=sum(
        prices[t] * model.discharge[t] - prices[t] * model.charge[t]
        for t in model.T
    ),
    sense=maximize
)

# Constraints
model.soc_constraints = ConstraintList()

for t in model.T:
    if t == 0:
        # First hour: start from initial SOC
        model.soc_constraints.add(
            model.soc[t] == INITIAL_SOC + EFFICIENCY * model.charge[t] - model.discharge[t]
        )
    else:
        # Every other hour: SOC carries over
        model.soc_constraints.add(
            model.soc[t] == model.soc[t-1] + EFFICIENCY * model.charge[t] - model.discharge[t]
        )

# Solve
solver = SolverFactory("appsi_highs")
result = solver.solve(model)

# Extract results
charge_schedule    = [value(model.charge[t])    for t in model.T]
discharge_schedule = [value(model.discharge[t]) for t in model.T]
soc_schedule       = [value(model.soc[t])       for t in model.T]

revenue = sum(prices[t] * discharge_schedule[t] - prices[t] * charge_schedule[t] for t in range(T))

print(f"Optimal daily revenue: {revenue:.4f} SEK/MWh")
print(f"\nHour | Price | Charge | Discharge | SOC")
print("-" * 50)
for t in range(T):
    print(f"  {t:02d} | {prices[t]:.3f} | {charge_schedule[t]:.3f}  | {discharge_schedule[t]:.3f}     | {soc_schedule[t]:.3f}")

Optimal daily revenue: 0.9776 SEK/MWh

Hour | Price | Charge | Discharge | SOC
--------------------------------------------------
  00 | 1.081 | 0.000  | 0.000     | 0.500
  01 | 1.041 | 0.000  | 0.000     | 0.500
  02 | 1.007 | 0.000  | 0.000     | 0.500
  03 | 0.976 | 0.500  | 0.000     | 0.950
  04 | 0.997 | 0.056  | 0.000     | 1.000
  05 | 1.024 | 0.000  | 0.000     | 1.000
  06 | 1.166 | 0.000  | -0.000     | 1.000
  07 | 1.235 | 0.000  | 0.500     | 0.500
  08 | 1.238 | 0.000  | 0.500     | 0.000
  09 | 1.098 | 0.000  | -0.000     | 0.000
  10 | 0.996 | 0.000  | -0.000     | 0.000
  11 | 0.892 | 0.000  | 0.000     | -0.000
  12 | 0.798 | 0.500  | 0.000     | 0.450
  13 | 0.795 | 0.500  | 0.000     | 0.900
  14 | 0.811 | 0.111  | 0.000     | 1.000
  15 | 0.855 | 0.000  | 0.000     | 1.000
  16 | 0.912 | 0.000  | -0.000     | 1.000
  17 | 1.006 | 0.000  | -0.000     | 1.000
  18 | 1.034 | 0.000  | -0.000     | 1.000
  19 | 1.132 | 0.000  | 0.500     | 0.500
  20 | 1.210 | 0.000  |

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

# Price line
fig.add_trace(go.Scatter(
    x=df_hourly["hour"], y=prices,
    name="Price (SEK/kWh)",
    line=dict(color="white", width=2)
))

# Charging bars (negative = buying)
fig.add_trace(go.Bar(
    x=df_hourly["hour"], y=[-c for c in charge_schedule],
    name="Charging (buy)",
    marker_color="royalblue",
    opacity=0.7
))

# Discharging bars (positive = selling)
fig.add_trace(go.Bar(
    x=df_hourly["hour"], y=discharge_schedule,
    name="Discharging (sell)",
    marker_color="crimson",
    opacity=0.7
))

# SOC line
fig.add_trace(go.Scatter(
    x=df_hourly["hour"], y=soc_schedule,
    name="State of Charge (MWh)",
    line=dict(color="lime", width=2, dash="dash"),
    yaxis="y2"
))

fig.update_layout(
    title=f"BESS Optimal Dispatch — SE3 | Daily Revenue: {revenue:.4f} SEK/MWh",
    template="plotly_dark",
    barmode="overlay",
    yaxis=dict(title="Price (SEK/kWh) / Power (MW)"),
    yaxis2=dict(title="SOC (MWh)", overlaying="y", side="right"),
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.2)
)

fig.show()

In [5]:
from datetime import datetime, timedelta
import time
import requests
import pandas as pd

def fetch_prices_for_date(date):
    url = f"https://www.elprisetjustnu.se/api/v1/prices/{date.year}/{date.strftime('%m-%d')}_SE3.json"
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()
    return None

# Fetch last 90 days
all_data = []
end_date = datetime.today() - timedelta(days=1)
start_date = end_date - timedelta(days=90)

current = start_date
while current <= end_date:
    data = fetch_prices_for_date(current)
    if data:
        all_data.extend(data)
    current += timedelta(days=1)
    time.sleep(0.05)

# Build DataFrame
df_hist = pd.DataFrame(all_data)
df_hist = df_hist[["time_start", "SEK_per_kWh"]]

# Fix: Convert to UTC to handle mixed Daylight Savings Time offsets
df_hist["time_start"] = pd.to_datetime(df_hist["time_start"], utc=True)
df_hist = df_hist.rename(columns={"time_start": "hour", "SEK_per_kWh": "price"})

# Set 'hour' as index for resampling
df_hist = df_hist.set_index("hour")

# Resample to hourly (handling the transition between 15-min and 60-min data if any)
df_hist = df_hist.resample("h").mean().reset_index()

print(f"Total hours: {len(df_hist)}")
print(df_hist.head())

Total hours: 2183
                       hour     price
0 2026-02-17 23:00:00+00:00  0.788007
1 2026-02-18 00:00:00+00:00  0.841542
2 2026-02-18 01:00:00+00:00  0.889602
3 2026-02-18 02:00:00+00:00  0.904410
4 2026-02-18 03:00:00+00:00  0.981793


In [6]:
# Feature engineering
df_feat = df_hist.copy()

# Time features
df_feat["hour_of_day"] = df_feat["hour"].dt.hour
df_feat["day_of_week"] = df_feat["hour"].dt.dayofweek
df_feat["month"] = df_feat["hour"].dt.month
df_feat["is_weekend"] = (df_feat["day_of_week"] >= 5).astype(int)

# Lag features (what the price was before)
df_feat["price_lag_1h"]  = df_feat["price"].shift(1)   # 1 hour ago
df_feat["price_lag_24h"] = df_feat["price"].shift(24)  # same hour yesterday
df_feat["price_lag_48h"] = df_feat["price"].shift(48)  # same hour 2 days ago

# Rolling averages
df_feat["price_roll_24h"] = df_feat["price"].rolling(24).mean()  # last 24h average
df_feat["price_roll_7d"]  = df_feat["price"].rolling(168).mean() # last 7 days average

# Drop rows with NaN from lag/rolling
df_feat = df_feat.dropna()

print(f"Dataset size after feature engineering: {len(df_feat)} rows")
print(df_feat.head())

Dataset size after feature engineering: 2016 rows
                         hour     price  hour_of_day  day_of_week  month  \
167 2026-02-24 22:00:00+00:00  1.003650           22            1      2   
168 2026-02-24 23:00:00+00:00  0.989513           23            1      2   
169 2026-02-25 00:00:00+00:00  0.919898            0            2      2   
170 2026-02-25 01:00:00+00:00  0.873575            1            2      2   
171 2026-02-25 02:00:00+00:00  0.870928            2            2      2   

     is_weekend  price_lag_1h  price_lag_24h  price_lag_48h  price_roll_24h  \
167           0      1.092475       0.957477       0.629458        1.267685   
168           0      1.003650       0.945770       0.617900        1.269508   
169           0      0.989513       0.933652       0.560277        1.268935   
170           0      0.919898       0.946145       0.556110        1.265911   
171           0      0.873575       0.942105       0.535065        1.262945   

     price_roll_7d

In [7]:
!pip install lightgbm -q

In [8]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Define features and target
features = [
    "hour_of_day", "day_of_week", "month", "is_weekend",
    "price_lag_1h", "price_lag_24h", "price_lag_48h",
    "price_roll_24h", "price_roll_7d"
]

X = df_feat[features]
y = df_feat["price"]

# Split — last 7 days as test, rest as train
# Important: time series split, no shuffling
split = len(X) - 168  # last 168 hours = 7 days

X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

# Train LightGBM
model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbose=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(50, verbose=False)]
)

# Evaluate
y_pred = model.predict(X_test)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print(f"MAE:  {mae:.4f} SEK/kWh")
print(f"RMSE: {rmse:.4f} SEK/kWh")
print(f"MAPE: {mape:.2f}%")

MAE:  0.0690 SEK/kWh
RMSE: 0.0903 SEK/kWh
MAPE: 10.89%


In [10]:
import plotly.graph_objects as go

# Get test period dates
test_dates = df_feat["hour"].iloc[split:].values

# Plot forecast vs actual
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=test_dates,
    y=y_test.values,
    name="Actual Price",
    line=dict(color="white", width=2)
))

fig.add_trace(go.Scatter(
    x=test_dates,
    y=y_pred,
    name="Forecasted Price",
    line=dict(color="royalblue", width=2, dash="dash")
))

fig.update_layout(
    title=f"LightGBM Price Forecast vs Actual — SE3 | MAPE: {mape:.2f}%",
    template="plotly_dark",
    yaxis_title="Price (SEK/kWh)",
    xaxis_title="Hour",
    hovermode="x unified"
)

fig.show()

In [13]:
from pyomo.environ import *

# Battery Parameters (Ensuring they are defined in this scope)
CAPACITY = 1.0      # MWh
MAX_POWER = 0.5     # MW
EFFICIENCY = 0.90   # 90%
INITIAL_SOC = 0.5   # 50%

# Use last 24 hours of forecast as tomorrow's price signal
forecast_prices = y_pred[-24:]
actual_prices   = y_test.values[-24:]

# Run optimizer on prices
def run_optimizer(prices):
    T = len(prices)
    model_opt = ConcreteModel()
    model_opt.T = RangeSet(0, T-1)
    model_opt.charge    = Var(model_opt.T, bounds=(0, MAX_POWER))
    model_opt.discharge = Var(model_opt.T, bounds=(0, MAX_POWER))
    model_opt.soc       = Var(model_opt.T, bounds=(0, CAPACITY))
    model_opt.obj = Objective(
        expr=sum(prices[t] * model_opt.discharge[t] - prices[t] * model_opt.charge[t]
                 for t in model_opt.T),
        sense=maximize
    )
    model_opt.soc_constraints = ConstraintList()
    for t in model_opt.T:
        if t == 0:
            model_opt.soc_constraints.add(
                model_opt.soc[t] == INITIAL_SOC + EFFICIENCY * model_opt.charge[t] - model_opt.discharge[t]
            )
        else:
            model_opt.soc_constraints.add(
                model_opt.soc[t] == model_opt.soc[t-1] + EFFICIENCY * model_opt.charge[t] - model_opt.discharge[t]
            )
    SolverFactory("appsi_highs").solve(model_opt)
    charge    = [value(model_opt.charge[t])    for t in model_opt.T]
    discharge = [value(model_opt.discharge[t]) for t in model_opt.T]
    return charge, discharge

# Optimize on forecast prices
charge_forecast, discharge_forecast = run_optimizer(forecast_prices)

# Optimize on actual prices (perfect foresight benchmark)
charge_perfect, discharge_perfect = run_optimizer(actual_prices)

# Calculate real revenue for both schedules using ACTUAL prices
revenue_forecast = sum(actual_prices[t] * discharge_forecast[t] - actual_prices[t] * charge_forecast[t] for t in range(24))
revenue_perfect  = sum(actual_prices[t] * discharge_perfect[t]  - actual_prices[t] * charge_perfect[t]  for t in range(24))
capture_rate     = (revenue_forecast / revenue_perfect) * 100

print(f"Revenue with perfect foresight: {revenue_perfect:.4f} SEK/MWh")
print(f"Revenue with ML forecast:       {revenue_forecast:.4f} SEK/MWh")
print(f"Capture rate:                   {capture_rate:.1f}%")

Revenue with perfect foresight: 0.9776 SEK/MWh
Revenue with ML forecast:       0.8897 SEK/MWh
Capture rate:                   91.0%


In [14]:
# Backtest over 90 days
results = []

# We need at least 168 hours of history before we can forecast
# So we start from day 7 and walk forward day by day

df_backtest = df_feat.reset_index(drop=True)

for day in range(7, len(df_backtest) // 24 - 1):
    # Get the 24 hours we want to optimize
    start_idx = day * 24
    end_idx   = start_idx + 24

    if end_idx > len(df_backtest):
        break

    # Historical data available up to this point
    history = df_backtest.iloc[:start_idx]

    if len(history) < 168:
        continue

    # Actual prices for this day
    actual_day = df_backtest.iloc[start_idx:end_idx]
    actual_prices_day = actual_day["price"].values

    if len(actual_prices_day) < 24:
        continue

    # Build forecast features for next 24 hours
    forecast_features = []
    for h in range(24):
        idx = start_idx + h
        if idx >= len(df_backtest):
            break
        row = df_backtest.iloc[idx]
        forecast_features.append({
            "hour_of_day"   : row["hour_of_day"],
            "day_of_week"   : row["day_of_week"],
            "month"         : row["month"],
            "is_weekend"    : row["is_weekend"],
            "price_lag_1h"  : df_backtest.iloc[idx-1]["price"]  if idx > 0   else 0,
            "price_lag_24h" : df_backtest.iloc[idx-24]["price"] if idx >= 24  else 0,
            "price_lag_48h" : df_backtest.iloc[idx-48]["price"] if idx >= 48  else 0,
            "price_roll_24h": df_backtest.iloc[max(0,idx-24):idx]["price"].mean(),
            "price_roll_7d" : df_backtest.iloc[max(0,idx-168):idx]["price"].mean(),
        })

    if len(forecast_features) < 24:
        continue

    X_day = pd.DataFrame(forecast_features)[features]
    forecast_day = model.predict(X_day)

    # Run optimizer on forecast
    charge_f, discharge_f = run_optimizer(forecast_day)

    # Run optimizer on perfect foresight
    charge_p, discharge_p = run_optimizer(actual_prices_day)

    # Revenue using actual prices
    rev_forecast = sum(actual_prices_day[t] * discharge_f[t] - actual_prices_day[t] * charge_f[t] for t in range(24))
    rev_perfect  = sum(actual_prices_day[t] * discharge_p[t] - actual_prices_day[t] * charge_p[t] for t in range(24))
    capture      = (rev_forecast / rev_perfect * 100) if rev_perfect > 0 else 0

    results.append({
        "date"            : actual_day["hour"].iloc[0],
        "revenue_forecast": rev_forecast,
        "revenue_perfect" : rev_perfect,
        "capture_rate"    : capture
    })

df_results = pd.DataFrame(results)

print(f"Days backtested:         {len(df_results)}")
print(f"Total revenue (forecast): {df_results['revenue_forecast'].sum():.4f} SEK/MWh")
print(f"Total revenue (perfect):  {df_results['revenue_perfect'].sum():.4f} SEK/MWh")
print(f"Average capture rate:     {df_results['capture_rate'].mean():.1f}%")
print(f"Best day capture rate:    {df_results['capture_rate'].max():.1f}%")
print(f"Worst day capture rate:   {df_results['capture_rate'].min():.1f}%")

Days backtested:         76
Total revenue (forecast): 95.9043 SEK/MWh
Total revenue (perfect):  100.3745 SEK/MWh
Average capture rate:     94.9%
Best day capture rate:    100.0%
Worst day capture rate:   78.5%


In [15]:
fig = go.Figure()

# Revenue bars
fig.add_trace(go.Bar(
    x=df_results["date"],
    y=df_results["revenue_forecast"],
    name="Forecast Revenue",
    marker_color="royalblue",
    opacity=0.8
))

fig.add_trace(go.Bar(
    x=df_results["date"],
    y=df_results["revenue_perfect"],
    name="Perfect Foresight Revenue",
    marker_color="gray",
    opacity=0.5
))

# Capture rate line
fig.add_trace(go.Scatter(
    x=df_results["date"],
    y=df_results["capture_rate"],
    name="Capture Rate (%)",
    line=dict(color="lime", width=2),
    yaxis="y2"
))

# Average capture rate reference line
fig.add_hline(
    y=df_results["capture_rate"].mean(),
    line_dash="dash",
    line_color="yellow",
    annotation_text=f"Avg Capture: {df_results['capture_rate'].mean():.1f}%",
    annotation_position="top left",
    yref="y2"
)

fig.update_layout(
    title="BESS Backtest — 76 Days | SE3 Nord Pool | LightGBM Forecast",
    template="plotly_dark",
    barmode="overlay",
    yaxis=dict(title="Daily Revenue (SEK/MWh)"),
    yaxis2=dict(
        title="Capture Rate (%)",
        overlaying="y",
        side="right",
        range=[0, 120]
    ),
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.2)
)

fig.show()